# 04 — Audio embeddings subset (Phase 2, Step 2)

Стримим `embeddings.parquet` (14 GB, 7.72M items) с HF Hub через `HfFileSystem` + `pyarrow`,
фильтруем по `item_id_to_idx.pkl` из Phase 1 (276,305 items) и сохраняем плотный
`[n_items+1, 128]` float32 массив в `artifacts/audio/embeddings.npy` (~135 MB).

Локально 14 GB качать не хочется — поэтому ноутбук рассчитан на **Colab** (95 GB RAM, быстрая сеть).
На диск пишется только результат.

## 0. Setup

In [ ]:
# Colab bootstrap (раскомментировать в Colab):
# from google.colab import userdata
# token = userdata.get('git')
# !git clone -b models-1 https://$token@github.com/Vladislavbro/music-recommendations.git
# %cd music-recommendations
# !pip install -q pyarrow huggingface_hub numpy

In [ ]:
import os, sys, pickle
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print('project root:', PROJECT_ROOT)

## 1. Загружаем item_id_to_idx из Phase 1

In [ ]:
ITEM_MAP_PATH = PROJECT_ROOT / 'artifacts' / 'gsasrec' / 'item_id_to_idx.pkl'
with open(ITEM_MAP_PATH, 'rb') as f:
    item_id_to_idx = pickle.load(f)

n_items = max(item_id_to_idx.values())
print(f'items: {len(item_id_to_idx):,}, max idx: {n_items:,}')
assert min(item_id_to_idx.values()) == 1, 'idx должен начинаться с 1 (0 = PAD)'

## 2. Стриминговая выгрузка subset

Идём по 30 row groups, фильтруем `item_id in target_ids` через `np.isin`,
декодируем `large_list<double>` → `[n_match, 128]` float32, кладём по `item_idx`.

По времени: на Colab ~5–10 мин (сеть и парсинг parquet).

In [ ]:
from src.data.audio_embeddings import extract_audio_subset

OUTPUT_PATH = PROJECT_ROOT / 'artifacts' / 'audio' / 'embeddings.npy'

embeds = extract_audio_subset(
    item_id_to_idx=item_id_to_idx,
    output_path=OUTPUT_PATH,
    use_normalized=False,
    verbose=True,
)

## 3. Sanity-check

In [ ]:
import numpy as np

arr = np.load(OUTPUT_PATH)
print('shape:', arr.shape, 'dtype:', arr.dtype)
print('PAD row (idx=0) all zeros:', not arr[0].any())

norms = np.linalg.norm(arr[1:], axis=1)
zero_rows = int((norms == 0).sum())
print(f'rows with zero norm (missing embedding): {zero_rows} / {arr.shape[0] - 1}')
print(f'norm stats (non-zero rows): mean={norms[norms > 0].mean():.4f}, '
      f'min={norms[norms > 0].min():.4f}, max={norms[norms > 0].max():.4f}')
print(f'NaN: {int(np.isnan(arr).sum())}, Inf: {int(np.isinf(arr).sum())}')
print(f'file size: {OUTPUT_PATH.stat().st_size / 2**20:.1f} MB')